In [0]:
%sql

CREATE OR REPLACE TABLE dev.claims_project.gold_provider_monthly AS
WITH monthly AS (
    SELECT
        ProviderID,
        ProviderSpecialty,
        ProviderLocation,
        DATE_TRUNC('MONTH', ClaimDate)
            AS service_month,
        COUNT(DISTINCT ClaimID) AS total_claims,
        COUNT(DISTINCT PatientID) AS total_patients,
        ROUND(SUM(ClaimAmount), 2) AS total_paid,
        ROUND(AVG(ClaimAmount), 2) AS avg_claim_amount,
        ROUND(AVG(claim_lag_days), 2) AS avg_claim_lag_days
    FROM dev.claims_project.silver_claims_enriched
    GROUP BY
        ProviderID,
        ProviderSpecialty,
        ProviderLocation,
        DATE_TRUNC('MONTH', ClaimDate)
),

windowed AS (
    SELECT
        *,
        LAG(total_paid) OVER (
            PARTITION BY ProviderID ORDER BY service_month
        ) AS previous_month_paid,

        total_paid
        -
        LAG(total_paid) OVER (
            PARTITION BY ProviderID ORDER BY service_month
        ) AS paid_change_from_previous_month,

        ROUND(
            100.0 *
            (
                total_paid
                -
                LAG(total_paid) OVER (
                    PARTITION BY ProviderID ORDER BY service_month
                )
            )
            /
            NULLIF(
                LAG(total_paid) OVER (
                    PARTITION BY ProviderID
                    ORDER BY service_month
                ), 0
            ), 2
        ) AS paid_percent_change,

        ROUND(
            AVG(total_paid) OVER (
                PARTITION BY ProviderID ORDER BY service_month
                ROWS BETWEEN 2 PRECEDING
                AND CURRENT ROW
            ),  2
        ) AS rolling_3_month_avg_paid,

        RANK() OVER (
            PARTITION BY
                service_month,
                ProviderLocation
            ORDER BY total_paid DESC
        ) AS provider_rank_in_location_month,

        NTILE(4) OVER (
            PARTITION BY service_month
            ORDER BY total_paid DESC
        ) AS national_paid_quartile
    FROM monthly
)
